# Pasientvennlige radiologirapporter med LLM

Vi skal oversette norske radiologirapporter til pasientvennlig språk ved hjelp av en språkmodell, og se hvordan ulike instruksjoner endrer hva modellen sier.

## Før dere starter

1. Logg inn med en Google-konto for å kjøre notatboken i Colab.
2. Kjør én og én celle med `Shift + Enter`.
3. Første gang får du en advarsel om at notatboken ikke er laget av Google. Trykk **Run anyway**.
4. Får du `NameError`, har Colab mistet forbindelsen. Velg *Runtime → Run before* og kjør cellen på nytt. Med Claude må nøkkelen limes inn på nytt.
5. Vil du beholde endringene dine, lagre en kopi (*File → Save a copy in Drive*).

## 1. Kom i gang

Notatboken bruker **Claude Sonnet 5.5** fra Anthropic. Den krever en API-nøkkel som dere får av kursholderen. Har dere ikke nøkkel, velg **Gemini 2.5 Flash**, som er gratis i Colab, men en eldre og svakere modell.

Velg modell i feltet `MODELL` og kjør cellen under. Med Claude kommer det opp et felt der du limer inn nøkkelen og trykker Enter. Nøkkelen skjules og lagres ikke i notatboken.

In [ ]:
#@title Velg modell og kjør cellen
MODELL = "Claude Sonnet 5.5"  # @param ["Claude Sonnet 5.5", "Gemini 2.5 Flash (gratis, ingen nøkkel)"]

import base64
import json
import re
import subprocess
import sys
from getpass import getpass

import requests
from IPython.display import Markdown, display

GEMINI_MODEL = "google/gemini-2.5-flash"
CLAUDE_MODEL = "claude-sonnet-5-5"
OPUS_MODEL = "claude-opus-5-5"  # brukes bare til bildet i del 10
# Anthropics standardnivå for hvor mye hver modell tenker før den svarer
EFFORT = {CLAUDE_MODEL: "high", OPUS_MODEL: "medium"}
USE_CLAUDE = MODELL.startswith("Claude")
MODELLNAVN = "Claude Sonnet 5.5" if USE_CLAUDE else "Gemini 2.5 Flash"
claude = None


def connect_claude():
    """Installer Anthropic-biblioteket, spør etter nøkkelen og sjekk at den virker."""
    pip = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "anthropic>=0.125"],
        capture_output=True, text=True,
    )
    if pip.returncode != 0:
        print(pip.stderr[-1000:])
    import anthropic

    # Fjern mellomrom og linjeskift som lett blir med når man limer inn
    key = "".join(getpass("Lim inn API-nøkkelen og trykk Enter: ").split())
    if not key:
        print("Du limte ikke inn noen nøkkel. Kjør cellen på nytt, eller velg Gemini.")
        return None
    client = anthropic.Anthropic(api_key=key, max_retries=4)
    try:
        # Å telle tokens er gratis, og sjekker både nøkkelen og modellen
        client.messages.count_tokens(
            model=CLAUDE_MODEL, messages=[{"role": "user", "content": "test"}]
        )
    except anthropic.AuthenticationError:
        print("Nøkkelen virker ikke. Sjekk at du har limt inn hele nøkkelen, og kjør cellen på nytt.")
        return None
    except anthropic.APIConnectionError:
        print("Fikk ikke kontakt med Anthropic. Sjekk internettforbindelsen og kjør cellen på nytt.")
        return None
    except anthropic.APIStatusError as e:
        print(f"Anthropic svarte med en feil ({e.status_code}): {e.message}")
        return None
    return client


if USE_CLAUDE:
    claude = connect_claude()
else:
    from google.colab import ai


def ask(user_text, system=None):
    """Send user_text (og en valgfri systemprompt) til modellen og returner svaret.

    Gemini i Colab tar én samlet tekst, så der legges systemprompten foran.
    Går noe galt, vises en kort feilmelding og svaret blir tomt.
    """
    if USE_CLAUDE:
        return _ask_claude(user_text, system)
    return _ask_gemini(user_text, system)


def _ask_gemini(user_text, system):
    full = f"{system}\n\n{user_text}" if system else user_text
    try:
        return ai.generate_text(full, model_name=GEMINI_MODEL)
    except Exception as e:
        _problem(f"Fikk ikke svar fra Gemini: {e}")
        return ""


def _ask_claude(user_text, system, model=CLAUDE_MODEL):
    import anthropic

    if claude is None:
        _problem("Claude er ikke klar. Kjør cellen «Velg modell» på nytt og lim inn nøkkelen.")
        return ""
    extra = {"system": system} if system else {}
    try:
        r = claude.beta.messages.create(
            model=model,
            max_tokens=16000,
            output_config={"effort": EFFORT[model]},
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
            messages=[{"role": "user", "content": user_text}],
            **extra,
        )
    except anthropic.AuthenticationError:
        _problem("Nøkkelen virker ikke lenger. Spør kursholderen.")
        return ""
    except anthropic.RateLimitError:
        _problem("Mange spør modellen samtidig. Vent et halvt minutt og kjør cellen på nytt.")
        return ""
    except anthropic.APIConnectionError:
        _problem("Fikk ikke kontakt med Claude. Sjekk internettforbindelsen og prøv igjen.")
        return ""
    except anthropic.APIStatusError as e:
        _problem(f"Claude svarte med en feil ({e.status_code}): {e.message}")
        return ""
    if r.stop_reason == "refusal":
        return "*Modellen ville ikke svare på denne forespørselen.*"
    return "".join(b.text for b in r.content if b.type == "text")


def ask_image(prompt, image_url, model="Claude Sonnet 5.5"):
    """Send et bilde og et spørsmål til Claude Sonnet 5.5 eller Claude Opus 5.5."""
    if not USE_CLAUDE:
        _problem("Bilder virker bare med Claude i denne notatboken. Se teksten under cellen.")
        return ""
    try:
        r = requests.get(
            image_url,
            headers={"User-Agent": "patient-friendly-llm-workshop (https://github.com/skaliy/patient-friendly-llm)"},
            timeout=20,
        )
        r.raise_for_status()
    except requests.RequestException:
        _problem("Fikk ikke lastet ned bildet. Prøv igjen om litt.")
        return ""
    image = {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": r.headers.get("Content-Type", "image/png").split(";")[0],
            "data": base64.standard_b64encode(r.content).decode("utf-8"),
        },
    }
    model_id = OPUS_MODEL if "Opus" in model else CLAUDE_MODEL
    return _ask_claude([image, {"type": "text", "text": prompt}], None, model_id)


def show(text):
    """Vis tekst formatert som markdown."""
    display(Markdown(text))


def _problem(text):
    """Vis en kort feilmelding i stedet for en lang feilutskrift."""
    show(f"**Feil:** {text}")


def _json_list(text):
    """Finn den første JSON-listen i et modellsvar. Returnerer None hvis ingen."""
    m = re.search(r"\[.*?\]", text, re.DOTALL)
    if not m:
        return None
    try:
        value = json.loads(m.group())
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, list) else None


def sml_search(term, limit=3):
    """Søk i Store medisinske leksikon (SML), akkurat slik søket på nettsiden gjør.

    Returnerer en liste med tupler (oppslagsord, definisjon, url),
    eller en tom liste hvis SML ikke svarer.
    """
    try:
        r = requests.get(
            "https://sml.snl.no/api/v1/search",
            params={"query": term, "limit": limit, "offset": 0},
            timeout=10,
        )
        r.raise_for_status()
        hits = r.json()
    except (requests.RequestException, ValueError):
        print(f"Fikk ikke svar fra SML for «{term}».")
        return []
    return [
        (h["headword"], h.get("first_sentences", ""), h["article_url"])
        for h in hits
    ]


def sml_lookup(term):
    """Finn den beste SML-artikkelen for en term.

    Foretrekker en artikkel med nøyaktig samme oppslagsord, og følger korte
    henvisninger (f.eks. "lungeemboli ... se lungeembolisme") til
    hovedartikkelen. Returnerer (oppslagsord, definisjon, url) eller None.
    """
    hits = sml_search(term, limit=5)
    if not hits:
        return None
    best = next((h for h in hits if h[0].lower() == term.lower()), hits[0])

    redirect = re.search(r"\bse ([^.,]+)\.?\s*$", best[1])
    if redirect and len(best[1]) < 150:
        target = redirect.group(1).strip()
        for h in sml_search(target, limit=5):
            if h[0].lower() == target.lower():
                return h
    return best


def extract_medical_terms(report):
    """Be modellen finne medisinske fagtermer som en pasient ikke forstår."""
    prompt = (
        "Trekk ut medisinske fagtermer fra denne radiologirapporten som en "
        "pasient sannsynligvis ikke forstår. Returner kun en JSON-liste med "
        'strenger, ingen forklaring. Eksempel: ["pneumothorax", "pleuravæske"]'
    )
    terms = _json_list(ask(report, system=prompt)) or []
    return [str(t) for t in terms]


def lookup_terms(terms):
    """Slå opp hver term i SML. Samme artikkel tas bare med én gang."""
    articles = []
    seen = set()
    for term in terms:
        hit = sml_lookup(term)
        if not hit:
            continue
        headword, definition, url = hit
        if not definition or url in seen:
            continue
        seen.add(url)
        articles.append(
            {"term": term, "article": headword, "definition": definition, "url": url}
        )
    return articles


def show_articles(articles):
    """Skriv ut en liste med SML-artikler: term, artikkel og starten av definisjonen."""
    if not articles:
        print("  (ingen)")
    for a in articles:
        short = a["definition"]
        if len(short) > 160:
            short = short[:160].rsplit(" ", 1)[0] + " ..."
        print(f"• {a['term']} → {a['article']}")
        print(f"  {short}")
    print()


def filter_relevant(report, articles, instructions):
    """La modellen velge hvilke SML-artikler som er relevante for rapporten.

    Hvis svaret ikke kan leses, beholdes alle artiklene.
    """
    if not articles:
        return []
    listing = "\n".join(
        f"{i}. {a['article']}: {a['definition']}" for i, a in enumerate(articles, 1)
    )
    numbers = _json_list(
        ask(f"Rapport:\n{report}\n\nArtikler:\n{listing}", system=instructions)
    )
    if numbers is None:
        print("Kunne ikke lese svaret fra modellen. Beholder alle artiklene.")
        return articles
    keep = {int(n) for n in numbers if str(n).strip().isdigit()}
    return [a for i, a in enumerate(articles, 1) if i in keep]


def translate_with_sml(report, system_prompt, articles):
    """Oversett en rapport med SML-definisjoner som ekstra hjelp."""
    context = "\n".join(f"- **{a['article']}**: {a['definition']}" for a in articles)
    message = f"""Rapport:
{report}

SML-kontekst (for å forstå fagtermene):
{context}

Skriv en pasientvennlig oversettelse."""
    return ask(message, system=system_prompt)


if not USE_CLAUDE or claude is not None:
    print(f"Klar. Notatboken bruker {MODELLNAVN}.")

### Test at modellen svarer

Får dere et kort svar under, virker alt.

In [ ]:
show(ask("Skriv dette med enkle ord: Ingen tegn til fraktur."))

## 2. Hva er en prompt?

En prompt er meldingen du sender til modellen. I dag bruker vi to typer samtidig:

**Systemprompt (instruksjoner)**
> Du oversetter radiologirapporter til pasientvennlig norsk. Bevar usikkerhet. Ikke legg til prognose ...

**Brukerprompt (innholdet)**
> CT thorax med kontrast. Det påvises ...

Systemprompten setter rammen: rolle, oppgave og hva modellen skal unngå. I chat-apper som ChatGPT er den skrevet av leverandøren og usynlig for deg. Brukerprompten er det konkrete innholdet, her radiologirapporten.

Vi skal skrive tre forskjellige systemprompter og se hva som skjer med samme rapport.

## 3. Materialet vi jobber med

Tre korte, **fiktive** radiologirapporter i samme stil som ekte rapporter.

In [ ]:
RAPPORT_1 = """CT thorax med intravenøs kontrast.
Det påvises en 18 mm rund fortetning i høyre overlapp, suspekt for malignitet.
Forstørrede lymfeknuter mediastinalt opp til 15 mm.
Ingen pleuravæske. Ingen tegn til lungeemboli."""

RAPPORT_2 = """MR caput.
Det ses en 8 mm hyperintens lesjon periventrikulært på FLAIR-sekvenser,
mest forenlig med kronisk iskemisk forandring.
Ingen patologisk kontrastopptak. Ingen tegn til akutt infarkt eller blødning."""

RAPPORT_3 = """Røntgen thorax.
Hjertestørrelse innen normalområde.
Det ses fortetning basalt i venstre lunge, kan passe med infiltrat.
Anbefaler klinisk korrelasjon. Ingen pneumothorax."""

RAPPORTER = {
    "Rapport 1 (CT thorax)": RAPPORT_1,
    "Rapport 2 (MR caput)": RAPPORT_2,
    "Rapport 3 (Røntgen thorax)": RAPPORT_3,
}

for navn, r in RAPPORTER.items():
    show(f"### {navn}")
    print(r)

## 4. V1: Naiv prompt

En naiv fremgangsmåte uten regler og eksempler.

In [ ]:
V1 = "Forklar denne radiologirapporten på et språk en pasient kan forstå."

show(ask(RAPPORT_1, system=V1))


### Diskusjonsoppgave (5 min, i grupper)

Kjør V1 på de andre rapportene også (cellene under) og se etter:

1. **Falsk trygghet.** Sier modellen at noe er "normalt" eller "ufarlig" når rapporten ikke sier det?
2. **Forsvunnet usikkerhet.** Fjerner modellen ord som *mistanke om*, *kan passe med* eller *suspekt for*?
3. **Tilført informasjon.** Legger modellen til prognose, årsaker eller symptomer som ikke står i rapporten?
4. **Manglende anbefaling.** Ignoreres "anbefaler klinisk korrelasjon" og liknende?

Noter ett konkret eksempel per rapport. Disse skal vi prøve å fikse i neste steg.

In [ ]:
show("### Rapport 2 (MR caput)")
show(ask(RAPPORT_2, system=V1))

In [ ]:
show("### Rapport 3 (Røntgen thorax)")
show(ask(RAPPORT_3, system=V1))

## 5. V2: Strukturert prompt med regler

En detaljert systemprompt med tydelige regler. Dette er det mange mener med "prompt engineering".

In [ ]:
V2 = """Du oversetter radiologiske rapporter til pasientvennlig norsk (bokmål).

Regler:
- Skriv på et nivå en pasient uten medisinsk bakgrunn kan forstå.
- Bevar usikkerhet nøyaktig. Hvis rapporten sier "mistanke om", "kan passe med",
  eller "ingen tegn til", skal dette gjenspeiles i oversettelsen.
- Gjengi kun det rapporten eksplisitt nevner. Ikke legg til prognose, mulige
  årsaker, eller symptomer.
- Beskriv aldri funn som "normale" med mindre rapporten sier det eksplisitt.
- Plasser fagbegrepet i parentes etter den enkle forklaringen
  (f.eks. "væske i lungene (lungeødem)").
- Bruk en nøytral, saklig tone i tredjeperson. Ikke skriv "du" eller "din".
- Svar bare med selve oversettelsen, uten overskrift eller innledning."""

show(ask(RAPPORT_1, system=V2))


### Sammenlign V1 og V2

Hva ble bedre? Hva er fortsatt feil?

Et vanlig mønster: V2 fikser de mest åpenbare problemene, men modellen bommer fortsatt på tone, glemmer parenteser, eller er ujevn fra rapport til rapport.

## 6. V3: Prompt med eksempler (*few-shot*)

Modellen treffer ofte bedre når den ser eksempler på ønsket svar (*few-shot*) enn når den bare får regler (*zero-shot*).

V3 er V2 pluss to ferdig løste eksempler.

In [ ]:
V3 = V2 + """

---
Eksempel 1:

Rapport:
CT abdomen/bekken med intravenøs kontrast. Blindtarmen fremstår utvidet med
tverrmål opp mot 13 mm. Det ses tydelig infiltrasjon i det periappendikulære
fettvevet, samt en appendikolitt basalt i lumen. Ingen tegn til fri væske.
Funnene er forenlige med akutt appendisitt.

Pasientvennlig oversettelse:
Undersøkelsen av buken viser tegn på akutt betennelse i blindtarmen (appendisitt). Den er forstørret,
og det er irritasjon i fettvevet rundt den. Det ses også en forkalkning
(appendikolitt) nederst i blindtarmen. Det er ikke tegn til væskeansamling i buken.

---
Eksempel 2:

Rapport:
MR columna lumbalis. I nivå L4/L5 påvises et venstresidig paramediant skiveprolaps
som komprimerer venstre L5-rot i recessen. Det er redusert skivehøyde i samme
nivå forenlig med degenerasjon.

Pasientvennlig oversettelse:
MR-bildene av korsryggen viser at skiven mellom 4. og 5. lendevirvel har presset
seg ut mot venstre side (prolaps). Prolapset trykker på en nerve i dette området.
Skiven i samme nivå har redusert høyde, noe som kan passe med slitasjeforandringer
(degenerasjon)."""

show(ask(RAPPORT_1, system=V3))


### Side om side: V1, V2, V3

Kjør cellen under for å se alle tre i rekkefølge på samme rapport.

In [ ]:
for navn, prompt in [
    ("V1 (naiv, én linje)", V1),
    ("V2 (regler, ingen eksempler)", V2),
    ("V3 (regler og to eksempler)", V3),
]:
    show(f"## {navn}")
    show(ask(RAPPORT_1, system=prompt))

### Diskusjonsoppgave (5 min)

- Hvor stor er forskjellen mellom V1 og V2?
- Hvor stor er forskjellen mellom V2 og V3?
- Hva er det eksemplene i V3 fanger som reglene i V2 ikke gjør?

## 7. Oppslag i SML

Hittil har modellen bare brukt det den er trent på. Hva skjer om vi i tillegg gir den norske fagtermer med definisjoner fra Store medisinske leksikon (SML)?

In [ ]:
#@title Vis diagram over stegene
# Tegn stegene fra rapport til endelig svar
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 2.6)
ax.axis("off")

boxes = [
    (0.2, "Rapport", "#E8F0FE", "black"),
    (2.2, "Modellen finner\nfagtermer", "#FCE8E6", "black"),
    (4.2, "Slå opp i\nSML", "#FEF7E0", "black"),
    (6.2, "Modellen fjerner\nirrelevante treff", "#F3E8FD", "black"),
    (8.2, "Definisjoner\nlegges til prompt", "#E6F4EA", "black"),
    (10.2, "Endelig\nsvar", "#4285F4", "white"),
]
w, h, y = 1.6, 1.2, 0.5

for x, text, color, text_color in boxes:
    ax.add_patch(
        patches.FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.05",
            linewidth=1.5, edgecolor="#5F6368", facecolor=color,
        )
    )
    ax.text(x + w / 2, y + h / 2, text,
            ha="center", va="center", fontsize=10, color=text_color)

# Piler mellom boksene
for (x1, *_), (x2, *_) in zip(boxes, boxes[1:]):
    ax.annotate("", xy=(x2, y + h / 2), xytext=(x1 + w, y + h / 2),
                arrowprops=dict(arrowstyle="->", lw=1.8, color="#5F6368"))

ax.text(6, 2.2,
        "Modellen tar to avgjørelser underveis: hvilke ord som trenger forklaring, og hvilke artikler som er relevante",
        ha="center", fontsize=10, style="italic", color="#5F6368")

plt.tight_layout()
plt.show()

### Steg 1: Søk i SML

Vi søker på to ord fra rapport 1 og henter de to beste treffene for hvert. Legg merke til hva som kommer opp for *malignitet*.

In [ ]:
for term in ["lungeemboli", "malignitet"]:
    print(f"Søk: {term}")
    for oppslagsord, definisjon, url in sml_search(term, limit=2):
        print(f"• {oppslagsord}")
        print(f"  Definisjon: {definisjon}")
        print(f"  URL: {url}")
    print()

### Steg 2: La modellen plukke ut termer fra en rapport

In [ ]:
termer = extract_medical_terms(RAPPORT_1)
print("Modellen mener disse termene trenger forklaring:")
for t in termer:
    print(f"  • {t}")


### Steg 3: Slå opp termene i SML

Handler alle artiklene om det rapporten beskriver?

In [ ]:
artikler = lookup_terms(termer)
show_articles(artikler)

### Steg 4: Relevansfilter

Søket finner artikler der ordet er nevnt, ikke nødvendigvis en artikkel om selve ordet. Sender vi alt videre, kan modellen ta med informasjon om helt andre tilstander.

Derfor lar vi modellen sortere listen først. Filteret er en ny systemprompt, bygget opp som de vi har skrevet tidligere.

In [ ]:
RELEVANSFILTER = """Du får en radiologirapport og en nummerert liste med artikler fra Store medisinske leksikon (SML).

Behold artikler som:
- forklarer et funn eller en tilstand som nevnes i rapporten, også når rapporten sier at den ikke er der (f.eks. "Ingen tegn til lungeemboli")
- beskriver anatomi der funnene er
- forklarer fagord pasienten trenger for å forstå rapporten

Fjern artikler om andre tilstander eller organer enn dem rapporten handler om.
Er du i tvil, behold artikkelen.

Returner kun en JSON-liste med numrene på artiklene du beholder, for eksempel [1, 3]."""

relevante = filter_relevant(RAPPORT_1, artikler, RELEVANSFILTER)

print("Beholdt:")
show_articles(relevante)
print("Fjernet:")
show_articles([a for a in artikler if a not in relevante])

### Steg 5: Sett det sammen

V3 tre ganger på samme rapport: uten SML, med alle treffene fra steg 3, og med bare de filtrerte artiklene fra steg 4.

In [ ]:
show("## V3 uten SML")
show(ask(RAPPORT_1, system=V3))

show("## V3 med alle SML-treff (uten filter)")
show(translate_with_sml(RAPPORT_1, V3, artikler))

show("## V3 med filtrerte SML-artikler")
show(translate_with_sml(RAPPORT_1, V3, relevante))

### Diskusjonsoppgave

- Velger modellen riktige termer? Faller noen viktige ut?
- Fjernet filteret noe som burde vært med, eller beholdt noe som burde vært fjernet?
- Tok modellen med noe fra de irrelevante artiklene i oversettelsen uten filter?
- Blir forklaringene mer presise, eller bare lengre?

## 8. Bonus: Tilpass for ulike mottakere

Samme rapport, forklart for ulike pasienter. Endre feltet `MOTTAKER` og kjør cellen.

In [ ]:
MOTTAKER = "en 70-åring uten medisinsk bakgrunn"  # @param {type:"string"}

V3_med_mottaker = V3 + f"\n\nMottaker: {MOTTAKER}. Tilpass språknivå deretter."

show(ask(RAPPORT_1, system=V3_med_mottaker))


Prøv for eksempel:

- "et barn på 14 år"
- "en person med høyere utdanning, men ikke medisinsk bakgrunn"
- "en pasient som allerede vet at de har lungekreft og kun vil høre om utviklingen"

**Det vanskelige spørsmålet:** Bør et klinisk verktøy gjøre denne tilpasningen automatisk? Hvem velger mottakerprofilen i et reelt system?

## 9. Avslutning: Modellen ser på et bilde

Språkmodeller kan også lese bilder. Vi sender røntgenbildet under til modellen to ganger: først med et enkelt spørsmål, så med rolle og et tydelig mål, slik vi har øvd på i dag.

![Skulderluksasjon, røntgen](https://upload.wikimedia.org/wikipedia/commons/e/ea/Luxation_epaule.PNG)

*Bildekilde: [Luxation epaule](https://commons.wikimedia.org/wiki/File:Luxation_epaule.PNG), Wikimedia Commons, MB, [CC BY-SA 2.5](https://creativecommons.org/licenses/by-sa/2.5/)*

In [ ]:
BILDEMODELL = "Claude Sonnet 5.5"  # @param ["Claude Sonnet 5.5", "Claude Opus 5.5"]
BILDE = "https://upload.wikimedia.org/wikipedia/commons/e/ea/Luxation_epaule.PNG"

show("## Enkelt spørsmål")
show(ask_image("Hva ser du på dette bildet?", BILDE, model=BILDEMODELL))

show("## Med rolle og mål")
show(ask_image(
    "Du er radiolog og skal forklare funn på dette bildet til en pasient uten medisinsk bakgrunn. "
    "Skriv to-tre setninger på enkelt norsk.",
    BILDE,
    model=BILDEMODELL,
))

Hva endret seg når modellen fikk en rolle og et mål?

Velg så Claude Opus 5.5 i feltet `BILDEMODELL` og kjør cellen igjen. Opus er større og dyrere enn Sonnet. Ble forklaringen bedre?

Bildet viser en skulder som har gått ut av ledd (skulderluksasjon). Fant modellen det?

Bruker dere Gemini, virker ikke cellen over. Lim da bildet inn i ChatGPT, Claude eller Gemini-appen og still de samme to spørsmålene der.